In [1]:
!wget https://raw.githubusercontent.com/alexeygrigorev/datasets/master/course_lead_scoring.csv

--2026-03-19 08:28:52--  https://raw.githubusercontent.com/alexeygrigorev/datasets/master/course_lead_scoring.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 80876 (79K) [text/plain]
Saving to: ‘course_lead_scoring.csv’

course_lead_scoring 100%[===================>]  78.98K  --.-KB/s    in 0.06s   

2026-03-19 08:28:53 (1.25 MB/s) - ‘course_lead_scoring.csv’ saved [80876/80876]



In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
df = pd.read_csv('course_lead_scoring.csv')
df.columns

Index(['lead_source', 'industry', 'number_of_courses_viewed', 'annual_income',
       'employment_status', 'location', 'interaction_count', 'lead_score',
       'converted'],
      dtype='str')

In [3]:
df.columns.isnull().sum()

np.int64(0)

In [4]:
df.isnull().sum()

lead_source                 128
industry                    134
number_of_courses_viewed      0
annual_income               181
employment_status           100
location                     63
interaction_count             0
lead_score                    0
converted                     0
dtype: int64

In [2]:
df[['lead_source','industry','employment_status','location']]=df[['lead_source','industry','employment_status','location']].fillna('NA').replace('','NA')

In [6]:
df.dtypes

lead_source                     str
industry                        str
number_of_courses_viewed      int64
annual_income               float64
employment_status               str
location                        str
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [3]:
numeric = ['number_of_courses_viewed','annual_income','interaction_count','lead_score','converted']
for c in numeric:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0)

In [4]:
df.isnull().sum()

lead_source                 0
industry                    0
number_of_courses_viewed    0
annual_income               0
employment_status           0
location                    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

In [13]:
df.industry.value_counts()

industry
retail           203
finance          200
other            198
healthcare       187
education        187
technology       179
manufacturing    174
NA               134
Name: count, dtype: int64

# Answer Question 1: retail

In [5]:
df[['number_of_courses_viewed','annual_income','interaction_count','lead_score']].corr()

,number_of_courses_viewed,annual_income,interaction_count,lead_score
number_of_courses_viewed,1.000000,0.009770,-0.023565,-0.004879
annual_income,0.009770,1.000000,0.027036,0.015610
interaction_count,-0.023565,0.027036,1.000000,0.009888
lead_score,-0.004879,0.015610,0.009888,1.000000


# option 1 - 0.009888
# option 2 - -0.004879
# option 3 - -0.023565
# option 4 - 0.027036
# Therefore Answer Question 2: annual_income and interaction_count

In [5]:
from sklearn.model_selection import train_test_split
df_train_full,df_test = train_test_split(df, test_size=0.2,random_state=42)
df_train,df_val = train_test_split(df_train_full, test_size=0.25,random_state=42)

y_train = df_train['converted'].values
y_val = df_val['converted'].values
y_test = df_test['converted'].values

del df_train['converted']
del df_val['converted']
del df_test['converted']
len(df_train), len(df_val), len(df_test), df['converted'].mean()

(876, 293, 293, np.float64(0.6190150478796169))

In [6]:
from sklearn.metrics import mutual_info_score
categorical =['lead_source','industry','employment_status','location']
numerical=['number_of_courses_viewed', 'annual_income','interaction_count', 'lead_score']

def mutual_info_convert_score(series):
    return mutual_info_score(series,y_train)

ms = df_train[categorical].apply(mutual_info_convert_score)
print(round(ms,2))

lead_source          0.04
industry             0.01
employment_status    0.01
location             0.00
dtype: float64


# Answer Question 3: lead_source

In [7]:
#one hot encoding
from sklearn.feature_extraction import DictVectorizer
df_train_dict = df_train[categorical+numerical].to_dict(orient='records')
dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(df_train_dict)
df_val_dict = df_val[categorical+numerical].to_dict(orient='records')
X_val = dv.transform(df_val_dict)

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train,y_train)
y_pred = model.predict_proba(X_val)[:,1]
print(y_pred)
churn_pred = (y_pred>=0.7)
(churn_pred == y_val).mean()
print(accuracy_score(y_val,churn_pred))

[0.61192163 0.79982617 0.53021344 0.47131479 0.57066132 0.44227169
 0.87127669 0.84883115 0.83290037 0.61497801 0.54968027 0.78153088
 0.69039786 0.77017122 0.5265944  0.91706425 0.53170635 0.42123049
 0.30146455 0.84881583 0.79488653 0.73670375 0.44527211 0.64838383
 0.4176882  0.75393418 0.90166116 0.33903049 0.43181431 0.9680681
 0.92018714 0.37487988 0.652301   0.90650057 0.75164117 0.64202121
 0.82250075 0.83375553 0.659116   0.30978853 0.78942264 0.35546366
 0.96517758 0.63389304 0.51274195 0.53230533 0.82287785 0.744074
 0.73452313 0.68955217 0.46964443 0.84539252 0.55635243 0.92637871
 0.65258021 0.61526273 0.63816995 0.28304018 0.48049824 0.57890618
 0.35497342 0.62175051 0.38960778 0.61156056 0.85304278 0.75430136
 0.89185954 0.71946459 0.95387623 0.89209517 0.75277088 0.33850139
 0.61376593 0.51622275 0.64088937 0.75978331 0.89312076 0.78234592
 0.57477802 0.5230894  0.75162464 0.45636646 0.81579585 0.79382273
 0.62186002 0.38948447 0.92007346 0.79436152 0.61193788 0.9264755